# 问题汇总
## 1. LangSmith原理？可以本地部署吗？
1. LangChain在开启LangSmith追踪后，会设置LangChainTracer回调，，在内部对应事件（比如模型调用前，响应后等）会触发异步调用，记录Run树(invoke()是root，内部其他调用是子Run)
2. 可以。官方提供Self-Hosted，支持Docker和K8S,但需要购买Enterprise License。
    github开源平替，LangFuse。
    ```shell
    git clone https://github.com/langfuse/langfuse.git
    cd langfuse
    docker compose up -d
    ```
    ```python
    from langfuse.langchain import CallbackHandler
    resp = chain.invoke(inputs, config={"callbacks": [CallbackHandler()]})
    ```

## 2. python基础：@classmethod和@property回顾？
1. `@classmethod`是类方法
2. `@property`是getter方法的设置

## 3. LangChain的管道命令是什么?
1. LangChain的管道命令实际是Runnable重写了__or__方法，在执行|时，会执行__or__方法,返回一个`RunnableSequence`对象, 上一个对象的输出作为下一个对象的输入

In [1]:
from html import parser

from common import init_simple_qwen_max
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from rich import print as rprint

qwen_max = init_simple_qwen_max()

prompt = ChatPromptTemplate(
    [
        ('system', '你是一个客服智能助手'),
        MessagesPlaceholder(variable_name='chat')
    ]
)

# 管道命令
# 等价于:
# prompt_value = prompt.invoke(...)
# qwen_max.invoke(prompt_value)

chat_chain = prompt | qwen_max

resp = chat_chain.invoke(
    {
        'chat': [
            ('user', '深圳天气怎么样？')
        ]
    }
)

rprint(resp)

AIMessage(
    content='我需要查询一下最新的天气信息才能回答您的问题。通常，您可以查看手机上的天气应用程序或访问气象网站来获取
最准确的深圳天气预报。如果您现在需要知道，我可以帮您查找最新的信息，请稍等片刻。\n\n（注：由于我当前无法直接访问互
联网以获取最新数据，建议您查看权威的天气预报网站或使用相应的天气应用查看深圳最新的天气情况。）',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 83,
            'prompt_tokens': 22,
            'total_tokens': 105,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'qwen-max',
        'system_fingerprint': None,
        'id': 'chatcmpl-60c52952-c6f7-9379-a0ca-e569c623a034',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f7509-b31b-7940-bb07-20895db6e78c-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 22,
        'output_tokens': 83,
        'total_tokens': 105,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {}
    }
)

## 4.为什么现在要拆成Anthropic和OpenAI两套API?
    1. 早期OpenAI火出圈，成为行业事实标准，国内大模型参考OpenAI的`/v1/chat/completions`，降低迁移成本，这样用户只需要修改`api_key`和`base_url`就可以切换模型，这是适配生态标准
    2. 而提供Anthropic是因为部分支持支Anthropic协议的工具爆火，它们不支持OpenAI协议，只支持Anthropic的`v1/messages`，为了自己模型能够在这些工具中使用，因此提供Anthropic协议，方便集成到这些工具，实际是一种解决方案。

## 5. 上下文长度是包括了输出和出入上下文长度吗？
1. 上下文长度=输出长度+输入长度。假设输入占了127k，输出1k就会被截断。
2. 部分厂商会拆开说明，限制输出长度和输入长度。

## 6. 为什么dashscope使用业务空间的专属域名，能够获得更高吞吐，降低时延？
    1. 公共域名`dashscope.aliyuncs.com`是所有人统一入口，可跨业务空间，而业务空间专属域名`{WorkspaceId}.{region}.maas.aliyuncs.com`直接转发至当前业务空间。
    2. 业务空间生产推荐原因：
        - 少一层多租户用户拥挤。专属业务空间域名把业务空间拆开，DNS/SNI很早就能将请求转发到具体的连接池、消息队列
        - 路由更直接。公共域名需要判断请求的业务空间转发，而专属域名不需要
        - 网络隔离。公共域名包含了很多杂流量，专属域名是纯净流量，网络高峰期也不会受到其他业务空间和杂流量影响

## 7. 模型怎么选型？如何比较性能？
    1. 选型：
        - 任务类型。
            - 多模态支持
            - 针对长文、RAG任务，不仅要看长下文窗口，还要看是否有长文理解能力
            - 如果频繁工具调用，选用Claude这类工具调用更稳定的
            - 代码编写任务，选用擅长代码编写的模型
            - 客服、简单对话，选用时延较低的
        - 成本考虑
            - 一开始可以选用性能中等、价格中等的模型
            - 简单任务用价格低的模型，复杂任务采用贵的模型
        - 合规
            - 数据是否出域，如果对数据安全要求高，不允许使用海外模型
    2. 比较性能：
        - 公开榜单粗筛：如 Arena、MMLU、LiveCodeBench、中文榜。可用来排除明显不行的，不能当最终依据。
        - 业务小样本评测：准备30-100条样本，同样提示词测试，评估维度：
            - 正确性：是否有幻觉，答案是否正确
            - 稳定性：工具调用成功率；多轮结果的方差
            - 延迟
            - 成本
            - 灰度A/B：灰度小部分流量，评估解决率、转人工率、客诉、工具调用成功率
        - 性价比计算公式：性价比 ≈ 任务成功率 / (平均时延 × 单次成本)。**怎么得出来的？**

## 8. TODO astream示例中和sleep实际并没有并发是吗？（）

## 10. Pydantic是什么？

## 9. TODO content_blocks的源码?
### 9.1 TODO content_blocks输入时，最终还是会把content_blocks转成content，那为什么有的模型通过content输入多模态无法识别，而content_blocks输入多模态就可以呢？是不是输入本身有问题，无关content和content_blocks

## 10. python基础：参数分隔符
    - / : 位置参数分隔符，控制在它之前的参数传递方式只能是位置传参
    - * : 关键字参数分隔符，控制在它之后的参数传递方式只能是关键字传参

## LangChain管道符

In [ ]:
# chain = prompt | model | parser
# resp = chain.invoke({'chat': '介绍一下星际穿越'})
# 等价于
# resp = parser.invoke(model.invoke(prompt.invoke({'chat': '介绍一下星际穿越'})))